# Writing and Running Your First Scripts

**Topic:** `1.8` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Reference implementations for all 10 exercises plus the robotics challenge.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

> **Compare, do not copy**
>
> Work each exercise yourself first. When you open this notebook, compare your approach with the reference: check the reasoning, the edge cases and the complexity. If your solution differs, that is fine — verify it with the tests provided and keep whichever is clearer.

## Solution 1 — Separate the program path from the arguments

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** sys.argv, slicing, lists

**Explanation and reasoning**

Slicing from index 1 handles every list length without a branch: an empty list and a single-element list both produce an empty result. Wrapping in list() gives the caller a new list, so mutating the result cannot corrupt sys.argv.

In [ ]:
def arguments(argv) -> list[str]:
    return list(argv[1:])

**Complexity**

Time O(n); space O(n).

**Edge cases and failure modes**

An empty input returns an empty list without raising.

**Alternative approaches**

argv[1:] alone is equivalent but shares the underlying list.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert arguments(['script.py', 'a', 'b']) == ['a', 'b']
assert arguments(['script.py']) == []
assert arguments([]) == []

---

## Solution 2 — Report the program name

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** pathlib, strings

**Explanation and reasoning**

Path.stem strips both the directory and the suffix in one call, so the same expression works on Windows and POSIX without touching separators. The `or 'python'` guard covers a path such as '/' whose stem is empty.

In [ ]:
from pathlib import Path


def program_name(argv) -> str:
    if not argv:
        return 'python'
    return Path(argv[0]).stem or 'python'

**Complexity**

Time O(n) in the path length; space O(n).

**Edge cases and failure modes**

A bare name such as 'roboctl' is returned unchanged.

**Alternative approaches**

os.path.splitext(os.path.basename(p)) is the older equivalent.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert program_name(['/opt/roboctl.py']) == 'roboctl'
assert program_name([]) == 'python'
assert program_name(['roboctl']) == 'roboctl'

---

## Solution 3 — Detect a boolean flag

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** membership, lists, validation

**Explanation and reasoning**

Membership on a list compares whole elements, which is exactly the semantics a command-line flag needs: '--verbose' must not match '--verbose-mode'. The empty-flag guard prevents `'' in args` from matching an empty argument, which is always accidental.

In [ ]:
def has_flag(args, flag: str) -> bool:
    if not isinstance(flag, str):
        raise TypeError('flag must be a string')
    if not flag:
        return False
    return flag in args

**Complexity**

Time O(n) in the list length; space O(1).

**Edge cases and failure modes**

An empty list returns False; a non-string flag raises TypeError.

**Alternative approaches**

args.count(flag) > 0 also works and reports the count.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert has_flag(['--verbose', '--name'], '--verbose') is True
assert has_flag(['--verbose'], '--verb') is False
assert has_flag([], '--verbose') is False

---

## Solution 4 — Map a status name to an exit code

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** dicts, error handling

**Explanation and reasoning**

The mapping is the contract, so it is written once as a module constant that a caller, a test and a shell script can all read. Indexing the dict raises KeyError for an unknown name, which is the correct signal for a programming error rather than a user error.

In [ ]:
EXIT_CODES = {'ok': 0, 'unknown': 1, 'usage': 2}


def exit_code_for(name: str) -> int:
    return EXIT_CODES[name]

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

A missing or misspelled name raises KeyError with the name in the message.

**Alternative approaches**

An Enum gives the same guarantee with clearer names in tracebacks.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert exit_code_for('ok') == 0
assert exit_code_for('usage') == 2

---

## Solution 5 — Build a usage line

**Difficulty:** `MEDIUM` &nbsp;|&nbsp; **Concepts:** strings, join, f-strings

**Explanation and reasoning**

An f-string composes the whole line in one expression, with the inner f-string producing the pipe-joined command list. Casting each command to str keeps the function working for non-string iterables without a special case.

In [ ]:
def usage_line(program: str, commands) -> str:
    return f"usage: {program} <{'|'.join(str(c) for c in commands)}>"

**Complexity**

Time O(n) in the command names; space O(n).

**Edge cases and failure modes**

An empty command list still produces the angle brackets.

**Alternative approaches**

Sorting the commands first makes the usage line deterministic.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert usage_line('roboctl', ['status', 'stop']) == 'usage: roboctl <status|stop>'
assert usage_line('p', []) == 'usage: p <>'

---

## Solution 6 — Split an inline option token

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** strings, partition, validation

**Explanation and reasoning**

partition splits on the first '=' only, so '--a=b=c' keeps 'b=c' as the value. Validating that the flag actually has a name after the dashes rejects the degenerate '--' case, and returning None for a bare flag lets the caller distinguish it from a valued option.

In [ ]:
def split_option(token: str) -> tuple[str, str | None]:
    if not isinstance(token, str) or not token.startswith('--'):
        raise ValueError(f'not a flag: {token!r}')
    name, separator, value = token.partition('=')
    if not separator:
        if len(name) < 3:
            raise ValueError(f'flag has no name: {token!r}')
        return name, None
    if not name[2:] or not value:
        raise ValueError(f'malformed option: {token!r}')
    return name, value

**Complexity**

Time O(n) in the token length; space O(n).

**Edge cases and failure modes**

'--name=' raises ValueError; a token without leading dashes raises ValueError.

**Alternative approaches**

argparse accepts both forms, and Module 9 is about that.

**Tests — run the cell to verify the reference solution**

In [ ]:
assert split_option('--name=rover-01') == ('--name', 'rover-01')
assert split_option('--verbose') == ('--verbose', None)
assert split_option('--a=b=c') == ('--a', 'b=c')

---

## Solution 7 — Parse flags and options with an index

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** loops, indexing, validation

**Explanation and reasoning**

Two paths advance the index differently - one for a boolean flag, two for an option with a value - and the explicit continue keeps that visible. Collecting every problem instead of stopping at the first means the user fixes all their mistakes in one pass.

In [ ]:
DEFAULTS = {'name': 'rover-01', 'count': 1, 'verbose': False}


def parse_args(args) -> tuple[dict, list[str]]:
    options = dict(DEFAULTS)
    errors: list[str] = []
    index = 0
    while index < len(args):
        token = args[index]
        if token == '--verbose':
            options['verbose'] = True
            index += 1
        elif token in {'--name', '--count'}:
            if index + 1 >= len(args):
                errors.append(f'{token} requires a value')
                index += 1
                continue
            value = args[index + 1]
            if token == '--name':
                options['name'] = value
            else:
                try:
                    count = int(value)
                except ValueError:
                    errors.append(f'--count must be an integer, got {value!r}')
                else:
                    if count < 0:
                        errors.append('--count must not be negative')
                    else:
                        options['count'] = count
            index += 2
            continue
        else:
            errors.append(f'unknown argument: {token!r}')
            index += 1
    return options, sorted(errors)

**Complexity**

Time O(n) in the arguments; space O(e) for errors.

**Edge cases and failure modes**

A missing value is reported and the loop still advances past the flag.

**Alternative approaches**

argparse does all of this and more; see Module 9.

**Tests — run the cell to verify the reference solution**

In [ ]:
options, errors = parse_args(['--verbose', '--name', 'rover-01'])
assert options == {'name': 'rover-01', 'count': 1, 'verbose': True}
assert errors == []
assert parse_args(['--count'])[1] != []

---

## Solution 8 — Run a child process and observe its result

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** subprocess, sys, streams

**Explanation and reasoning**

Passing sys.executable guarantees the child uses the same interpreter as the parent, which is the single most common source of confusion when a package appears to be installed but is not visible. Appending args after the -c source puts them where the child's sys.argv expects them.

In [ ]:
import subprocess
import sys


def run_python(code: str, args=(), timeout: float = 5.0) -> tuple[int, str, str]:
    result = subprocess.run(
        [sys.executable, '-c', code, *args],
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    return result.returncode, result.stdout, result.stderr

**Complexity**

Time is dominated by interpreter start-up in the child; space O(1).

**Edge cases and failure modes**

A timeout raises TimeoutExpired rather than hanging the caller.

**Alternative approaches**

Writing the code to a temporary file is slower but easier to debug.

**Tests — run the cell to verify the reference solution**

In [ ]:
code, out, err = run_python('import sys; print(sys.argv[1])', ['hello'])
assert code == 0 and out.strip() == 'hello'
bad = run_python('raise SystemExit(1)')
assert bad[0] == 1

---

## Solution 9 — Dispatch a command and return an exit status

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** dispatch, streams, exit codes

**Explanation and reasoning**

Validating the command against a tuple before branching means an unknown command is rejected in one place rather than falling through to a final else that has to guess. Every print names its stream, so a test can capture both and assert the contract.

In [ ]:
import sys

EXIT_OK, EXIT_UNKNOWN, EXIT_USAGE = 0, 1, 2
COMMANDS = ('status', 'stop', 'help')


def dispatch(argv, out=sys.stdout, errors=sys.stderr) -> int:
    if not argv:
        print('usage: roboctl <status|stop|help>', file=errors)
        return EXIT_USAGE
    command = argv[0]
    if command not in COMMANDS:
        print(f'unknown command: {command!r}', file=errors)
        return EXIT_UNKNOWN
    if command == 'help':
        print('usage: roboctl <status|stop|help>', file=out)
    else:
        print(f'{command}: ok', file=out)
    return EXIT_OK

**Complexity**

Time O(1); space O(1).

**Edge cases and failure modes**

An empty argv returns the usage code and writes nothing to stdout.

**Alternative approaches**

A dict of command functions removes the if-chain entirely.

**Tests — run the cell to verify the reference solution**

In [ ]:
import io
out, err = io.StringIO(), io.StringIO()
assert dispatch(['status'], out, err) == 0
assert dispatch([], out, err) == 2
assert dispatch(['nope'], out, err) == 1

---

## Solution 10 — Prove a module is import-safe

**Difficulty:** `HARD` &nbsp;|&nbsp; **Concepts:** import, __name__, subprocess

**Explanation and reasoning**

Seeding the namespace with __name__ set to something other than '__main__' reproduces exactly what the interpreter does on import, so the main guard correctly skips its block. Executing the source rather than importing it keeps the test hermetic and leaves no module behind.

In [ ]:
import io
from contextlib import redirect_stdout


def import_safety(source: str) -> str:
    """Execute source as if imported, and return what it printed."""
    buffer = io.StringIO()
    namespace = {'__name__': 'imported_module'}
    with redirect_stdout(buffer):
        exec(compile(source, '<module>', 'exec'), namespace)
    return buffer.getvalue()

**Complexity**

Time proportional to the source; space proportional to the output.

**Edge cases and failure modes**

Source with no top-level print returns an empty string.

**Alternative approaches**

Running the file in a subprocess is the end-to-end version of this test.

**Tests — run the cell to verify the reference solution**

In [ ]:
out = import_safety("print('top level')\nif __name__ == '__main__':\n    print('main only')")
assert out == 'top level\n'
assert 'main only' not in out

---

# Robotics challenge solution — ROBO-X Challenge: Fleet Bring-Up Tool

**Explanation**

Separating render from cli means the line a supervisor parses can be asserted on directly, while cli owns the streams and the status. Nothing here touches sys.argv, so the whole tool is a pure function of its arguments and a test needs neither a shell nor hardware.

In [ ]:
import io
import sys
from pathlib import Path

_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'shared').is_dir()), None)
if _root is not None:
    sys.path.insert(0, str(_root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

EXIT_OK, EXIT_UNKNOWN, EXIT_USAGE = 0, 1, 2
COMMANDS = ('status', 'stop', 'help')


def render(robot) -> str:
    state = robot.status()
    return f"{state['name']}:{state['battery_pct']:.1f}"


def cli(argv, out=sys.stdout, errors=sys.stderr) -> int:
    if not argv:
        print('usage: roboctl <status|stop|help>', file=errors)
        return EXIT_USAGE
    command = argv[0]
    if command not in COMMANDS:
        print(f'unknown command: {command!r}', file=errors)
        return EXIT_UNKNOWN
    if command == 'help':
        print('usage: roboctl <status|stop|help>', file=out)
        return EXIT_OK
    robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
    print(render(robot) if command == 'status' else 'stopped', file=out)
    return EXIT_OK

**Complexity**

Time O(1); space O(1).

**Notes and trade-offs**

An unknown command writes only to the error stream and returns 1.

In [ ]:
out, err = io.StringIO(), io.StringIO()
assert cli(['status'], out, err) == 0
assert 'rover-01' in out.getvalue()
assert cli(['nope'], out, err) == 1

## Reflection

For each exercise where your solution differed from the reference, write one sentence explaining *why* yours was better or worse. This is the highest-value part of the notebook:

- A case the reference handles but mine did not.
- A case mine handles better — and what the reference is missing.
- A performance difference that matters at robot scale.